In [1]:
import mpramnist
from mpramnist.DNASynBench.dataset import DNASynDataset

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights
from mpramnist.trainers import LitModel_DeepPromoter

from mpramnist import transforms as t
from mpramnist import target_transforms as t_t

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

In [2]:
BATCH_SIZE = 128
NUM_WORKERS = 8
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Define set of transforms
train_transform = t.Compose(
    [
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
val_test_transform = t.Compose(
    [  # probability of reverse-complement = 0.
        t.Seq2Tensor(),
    ]
)

In [4]:
args_2 = {'length':200,
    'n_seqs':100000,
    'motif':'GCCACGTGGC',
    'gc_content':0.41,
    'min_num':0,
    'max_num':5,
    'random_state':42}

In [5]:
train_dataset = DNASynDataset(
    split="train", transform=train_transform, args=args_2
)

val_dataset = DNASynDataset(
    split="val", transform=val_test_transform, args=args_2
)

test_dataset = DNASynDataset(
    split="test", transform=val_test_transform, args=args_2
)

Generating motifs...


100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [00:05<00:00, 19417.87it/s]


Shuffling...
Generating motifs...


100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [00:05<00:00, 19563.27it/s]


Shuffling...
Generating motifs...


100%|████████████████████████████████████████████████████████████████████████| 100000/100000 [00:05<00:00, 19611.63it/s]


Shuffling...


In [6]:
# encapsulate data into dataloader form
train_loader = data.DataLoader(
    dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)

val_loader = data.DataLoader(
    dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

test_loader = data.DataLoader(
    dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)

In [7]:
in_channels = len(train_dataset[0][0])
out_channels = 1

In [8]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_DeepPromoter(
    model=model, loss=nn.MSELoss(), weight_decay=1e-4, lr=1e-3, print_each=5
)

In [9]:
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[0],
    max_epochs=25,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/vastre34/miniconda3/envs/mpramnist/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [10]:
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.
/home/vastre34/miniconda3/envs/mpramnist/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

  | Name          | Type            | Params | Mode 
----------------------------------------------------------
0 | model         | HumanLegNet     | 1.3 M  | train
1 | loss          | MSELoss         | 0      | train
2 | train_pearson | PearsonCorrCoef | 0      | train
3 | val_pearson   | Pea

Training: |                                                                                       | 0/? [00:00…

/home/vastre34/miniconda3/envs/mpramnist/lib/python3.12/site-packages/lightning/pytorch/core/module.py:1293: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate


Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

`Trainer.fit` stopped: `max_epochs=25` reached.



--------------------------------------------------------------------------------
| Epoch: 24 | Val Loss: 0.00003 | Val Pearson: 0.99998 | Train Pearson: 0.99987 
--------------------------------------------------------------------------------



In [11]:
trainer.test(seq_model, dataloaders=test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing: |                                                                                        | 0/? [00:00…

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss          2.837137071765028e-05
      test_pearson          0.9999842047691345
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 2.837137071765028e-05, 'test_pearson': 0.9999842047691345}]